# Fruit Detection v3 - Training Notebook

**Model:** YOLOv8m (25.9M params)  
**Dataset:** 11,709 balanced training images across 8 fruit classes  
**Batch size:** Auto-detected (`batch=-1`) - uses max that fits in GPU VRAM  
**Expected training time:** ~1.5 hrs (T4) | ~55 min (A100)  

### Classes
| Index | Class | Index | Class |
|---|---|---|---|
| 0 | apple | 4 | pineapple |
| 1 | banana | 5 | watermelon |
| 2 | orange | 6 | grapes |
| 3 | mango | 7 | pomegranate |

---
## Step 0: Runtime Check
Make sure you are using **GPU Runtime**: Runtime -> Change runtime type -> T4 GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU')

## Step 1: Install Dependencies

In [ ]:
# Install ultralytics (includes YOLOv8)
!pip install ultralytics==8.3.* -q

# Verify
from ultralytics import YOLO
import ultralytics
print('Ultralytics version:', ultralytics.__version__)

## Step 2: Upload Dataset

Upload `dataset_v3.zip` from your local machine.  
**To create the zip:** Run this on your PC first:
```powershell
# In PowerShell from the project folder:
python export_for_colab.py
# Then upload dataset_v3_colab.zip to Colab
```

In [ ]:
from google.colab import files
import os

# Option A: Upload from your computer
print('Upload dataset_v3_colab.zip when prompted...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'Uploaded: {zip_name}')

In [ ]:
# Alternative: Mount Google Drive and copy from there
# Uncomment if you uploaded to Drive instead:

# from google.colab import drive
# drive.mount('/content/drive')
# zip_name = '/content/drive/MyDrive/dataset_v3_colab.zip'
# print('Using Drive file:', zip_name)

In [ ]:
import zipfile, os

DATASET_DIR = '/content/fruit_dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

print(f'Extracting {zip_name} to {DATASET_DIR}...')
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(DATASET_DIR)

# List what we got
for root, dirs, files_list in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')

print('Extraction complete!')

## Step 3: Configure Data YAML

In [ ]:
import yaml
from pathlib import Path

DATASET_DIR = '/content/fruit_dataset'

# Find the dataset_v3 folder inside the extracted zip
dataset_v3 = None
for p in Path(DATASET_DIR).rglob('train'):
    if (p / 'images').exists():
        dataset_v3 = p.parent
        break

if dataset_v3 is None:
    raise RuntimeError(f'Could not find dataset_v3 inside {DATASET_DIR}. Check zip structure.')

print('Dataset root found:', dataset_v3)

# Count images per split
for split in ['train', 'valid', 'test']:
    imgs = list((dataset_v3 / split / 'images').glob('*.jpg')) + \
           list((dataset_v3 / split / 'images').glob('*.png'))
    print(f'  {split}: {len(imgs)} images')

# Write the data.yaml with absolute paths (required for Colab)
YAML_PATH = '/content/data_v3.yaml'
data_cfg = {
    'path': str(dataset_v3),
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 8,
    'names': {
        0: 'apple', 1: 'banana', 2: 'orange', 3: 'mango',
        4: 'pineapple', 5: 'watermelon', 6: 'grapes', 7: 'pomegranate'
    }
}

with open(YAML_PATH, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f'\ndata.yaml written to: {YAML_PATH}')
print('\nContents:')
!cat /content/data_v3.yaml

## Step 4: Train YOLOv8m

Training config:
- **Model:** YOLOv8m (25.9M params - best speed/accuracy tradeoff on T4)
- **Batch:** `batch=-1` (Ultralytics AUTO-BATCH)
  - Finds the largest batch that fits in VRAM automatically
  - T4 (16 GB) -> ~24 | V100 (16 GB) -> ~24 | A100 (40 GB) -> ~64
  - Bigger batch = faster training + more stable gradients
- **cache='ram':** Pre-loads all images into RAM - eliminates disk I/O between epochs
- **Epochs:** 120 with early stopping (patience=25)
- **Augmentation:** Webcam-optimised (dark rooms, partial occlusion, varying distances)
- **Label smoothing:** 0.1 (prevents overconfidence)
- **Cosine LR:** Better convergence than step decay

In [ ]:
from ultralytics import YOLO
import torch

# PyTorch compatibility patch
_orig_load = torch.load
def _patched_load(*a, **kw):
    kw['weights_only'] = False
    return _orig_load(*a, **kw)
torch.load = _patched_load

YAML_PATH  = '/content/data_v3.yaml'
MODEL_NAME = 'fruit_v3'

# Show what GPU we have before training starts
print('GPU :', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('Auto-batch will select the largest batch that fits in VRAM...')

model = YOLO('yolov8m.pt')

results = model.train(
    data=YAML_PATH,
    epochs=120,
    imgsz=640,
    batch=-1,             # AUTO-BATCH: finds max batch size for the GPU
                          # T4->~24  V100->~24  A100->~64  (beats hardcoded 16)
    device='0',
    workers=4,            # Linux Colab: 4 workers is optimal
    patience=25,
    cache='ram',          # pre-load ALL images into RAM: no disk I/O per epoch
    project='/content/runs',
    name=MODEL_NAME,
    exist_ok=True,

    # Quality settings
    label_smoothing=0.1,  # prevents overconfidence
    cos_lr=True,          # cosine LR schedule
    close_mosaic=10,      # disable mosaic for last 10 epochs
    save_period=10,       # checkpoint every 10 epochs
    val=True,

    # Webcam-optimised augmentations
    hsv_h=0.020,
    hsv_s=0.80,
    hsv_v=0.50,
    degrees=12,
    translate=0.12,
    scale=0.60,
    shear=4.0,
    perspective=0.0003,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    erasing=0.40,
)

print('Training complete!')
print(f'Best weights: /content/runs/{MODEL_NAME}/weights/best.pt')

## Step 5: Evaluate on Test Set

In [ ]:
best_pt = f'/content/runs/{MODEL_NAME}/weights/best.pt'
eval_model = YOLO(best_pt)

metrics = eval_model.val(
    data=YAML_PATH,
    split='test',
    conf=0.25,
    iou=0.50,
)

print('\n=== TEST SET RESULTS ===')
print(f'mAP@50:     {metrics.box.map50:.3f}')
print(f'mAP@50-95:  {metrics.box.map:.3f}')
print(f'Precision:  {metrics.box.mp:.3f}')
print(f'Recall:     {metrics.box.mr:.3f}')

classes = ['apple','banana','orange','mango','pineapple','watermelon','grapes','pomegranate']
if hasattr(metrics.box, 'ap_class_index'):
    print('\nPer-class AP@50:')
    for idx, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
        print(f'  {classes[idx]:15s}  {ap:.3f}')

## Step 6: Download the Best Weights

In [ ]:
import shutil
from google.colab import files

best_pt = f'/content/runs/{MODEL_NAME}/weights/best.pt'

# Copy to easy location
shutil.copy(best_pt, '/content/fruit_v3_best.pt')

print('Downloading fruit_v3_best.pt ...')
files.download('/content/fruit_v3_best.pt')
print('Done! Place this file in your project models/ folder.')

## Step 7 (Optional): Quick Inference Test

Upload a test image to verify the model works.

In [ ]:
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

best_pt = f'/content/runs/{MODEL_NAME}/weights/best.pt'
inf_model = YOLO(best_pt)

# Upload a test image
from google.colab import files
up = files.upload()
test_img = list(up.keys())[0]

results = inf_model(test_img, conf=0.15)
result_img = results[0].plot()

plt.figure(figsize=(10, 8))
plt.imshow(result_img[..., ::-1])  # BGR -> RGB
plt.axis('off')
plt.title('Fruit Detection Results', fontsize=16)
plt.tight_layout()
plt.show()

for r in results:
    if r.boxes:
        classes = ['apple','banana','orange','mango','pineapple','watermelon','grapes','pomegranate']
        for box in r.boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            print(f'  {classes[cls]:15s}  conf={conf:.2f}')
    else:
        print('No detections. Try lowering conf threshold.')